Importing Dependencies

In [24]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, ServiceContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import AutoTokenizer, AutoModelForCausalLM
import wikipediaapi
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings
import os, glob
from pathlib import Path
from llama_index.core.node_parser import SentenceSplitter
from sentence_transformers import SentenceTransformer, util
import numpy as np

Set up Wikipedia API and fetch articles

In [22]:
wiki = wikipediaapi.Wikipedia(language='en', user_agent='Komal_Samuel_RAG_Project')

animals = ["Elephant", "Cheetah", "Giraffe", "Dolphin", "Penguin", "Kangaroo", "Panda", "Tiger", "Zebra", "Hippo"]

data = []
for animal in animals:
    page = wiki.page(animal)
    if page.exists():
        text = page.text
        data.append({"animal": animal, "text": text})
        with open(f"docs/{animal}.txt", "w", encoding="utf-8") as f:
            f.write(text)

2025-11-02 21:46:20,142 - INFO - Wikipedia: language=en, user_agent: Komal_Samuel_RAG_Project (Wikipedia-API/0.8.1; https://github.com/martin-majlis/Wikipedia-API/), extract_format=ExtractFormat.WIKI
2025-11-02 21:46:20,154 - INFO - Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Elephant&inprop=protection|talkid|watched|watchers|visitingwatchers|notificationtimestamp|subjectid|url|readable|preload|displaytitle|varianttitles
2025-11-02 21:46:20,687 - INFO - Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=extracts&titles=Elephant&explaintext=1&exsectionformat=wiki
2025-11-02 21:46:20,781 - INFO - Request URL: https://en.wikipedia.org/w/api.php?format=json&redirects=1&action=query&prop=info&titles=Cheetah&inprop=protection|talkid|watched|watchers|visitingwatchers|notificationtimestamp|subjectid|url|readable|preload|displaytitle|varianttitles
2025-11-02 21:46:20,984 - INFO - Request URL: https:

Simple test with model

In [ ]:
import ollama

response = ollama.chat(model='llama2', messages=[
  {'role': 'user', 'content': 'Explain retrieval augmented generation in one sentence.'}
])

print(response['message']['content'])

Load the model in with its tokenizer

In [ ]:
# run 'ollama serve' and 'ollama pull llama2' in terminal before running this code
llm = Ollama(model="llama2")

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

Settings.num_output = 256
Settings.context_window = 3000

2025-11-02 22:00:12,258 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Helper function to generate responses for different chunk sizes

In [28]:
def generate_responses(query):
    documents = SimpleDirectoryReader("docs").load_data()

    chunk_sizes = [128, 256, 512, 1024]

    for chunk_size in chunk_sizes:
        print(f"\n=== Chunk size: {chunk_size} ===")

        # Create a sentence-based text splitter
        splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=20) 
        nodes = splitter.get_nodes_from_documents(documents)

        Settings.llm = llm
        Settings.embed_model = embed_model

        index = VectorStoreIndex(nodes)
        query_engine = index.as_query_engine()

        response = query_engine.query(query)
        print("Response:", response.response)


Testing RAG retrieval for different queries

In [29]:
generate_responses("Explain how lions hunt together.")
generate_responses("What do elephants eat?")
generate_responses("Describe the habitat of penguins.")
generate_responses("How fast can a cheetah run?")


=== Chunk size: 128 ===


2025-11-02 22:02:35,415 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2025-11-02 22:02:55,597 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: Lions are known to hunt together as a coordinated effort, often relying on their senses of sight and hearing to locate prey. When they have located potential prey, they will stalk and surround it, working together to corner and subdue it. This collective hunting behavior is an important aspect of lion social dynamics and plays a crucial role in their survival and success as a species.

=== Chunk size: 256 ===


2025-11-02 22:04:38,802 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: Lions do not have a hunting strategy that involves them working together in the same way that tigers do. While lions are also ambush predators, they typically hunt alone or in small groups, rather than as part of a larger family unit. In fact, lions are known to be relatively solitary animals, with males often living and hunting alone outside of the breeding season. During this time, males will come together to mate with females, but afterward, they will go their separate ways until the next mating season.

Lions are skilled hunters and use a variety of tactics to catch their prey. They are known for their stealth and speed, and can move quietly and quickly over long distances in pursuit of their quarry. Once they have identified potential prey, they will stalk it silently and then make a sudden charge when they are close enough to make a kill. This type of hunting is often referred to as "stalking and pounce."

While lions do not hunt together in the same way that tigers do,

...

Response: Based on the provided context information, the maximum speed recorded for a cheetah is approximately 100.1 km/h (58 mph) sustained for 1-2 seconds, as mentioned in the text.

=== Chunk size: 512 ===


2025-11-02 22:19:33,297 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: According to the provided context information, the top speed of a cheetah has been recorded at 100.1 km/h (62.1 mph) for one individual. Additionally, other studies have measured the running speed of cheetahs as follows:

* Average running speed of cheetahs during hunts is estimated to be around 58 mph (93 km/h) based on GPS-IMU collars.
* The maximum sustained speed of cheetahs has been recorded at 104 km/h (65 mph) over a marked course.

Therefore, the fastest speed that a cheetah can run is approximately 100.1 km/h (62.1 mph), but they can sustain speeds of up to 104 km/h (65 mph) in short bursts.

=== Chunk size: 1024 ===


2025-11-02 22:19:56,922 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Response: Based on the provided context information, the fastest speed of a cheetah has been measured at approximately 104 km/h (65 mph) over a marked 201 m (220 yd) course. This was achieved through three runs, starting from behind a line and including measurements for both maximum sustained speed and running speed. The information provided in the context suggests that the cheetah is capable of reaching incredible speeds, with estimates suggesting it can run up to 150 km/h (93 mph) for short distances. However, it's important to note that these measurements were taken in controlled conditions and may not reflect the cheetah's actual running speed in real-world scenarios.


Evaluating chunk sizes with retrieval accuracy

In [ ]:
DATA_DIR = "./docs"  

doc_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")))

docs = []
doc_names = []
for p in doc_paths:
    with open(p, "r", encoding="utf-8") as f:
        docs.append(f.read())
        doc_names.append(Path(p).stem)  # file name without .txt = doc id

print(f"Loaded {len(docs)} documents from {DATA_DIR}")
print("Docs:", doc_names)

queries = [f"What does a {name} eat?" for name in doc_names]
qrels = [{i} for i in range(len(docs))]  # each query should match its corresponding doc

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def evaluate_chunk_size(chunk_size, overlap):
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=overlap)

    chunks, parents = [], []
    for i, doc in enumerate(docs):
        for chunk in splitter.split_text(doc):
            chunks.append(chunk)
            parents.append(i)

    chunk_embs = embed_model.encode(chunks, convert_to_tensor=True)

    correct = 0
    for qi, query in enumerate(queries):
        q_emb = embed_model.encode(query, convert_to_tensor=True)
        scores = util.cos_sim(q_emb, chunk_embs)[0]
        best_idx = int(np.argmax(scores))
        pred_doc = parents[best_idx]
        if pred_doc in qrels[qi]:
            correct += 1

    return correct / len(queries)

chunk_sizes = [128, 256, 512, 1024]
overlap = 0

print("Chunk Size | Accuracy")
print("---------------------")
for cs in chunk_sizes:
    acc = evaluate_chunk_size(cs, overlap)
    print(f"{cs:<10} | {acc:.2f}")

2025-11-02 21:46:35,161 - INFO - Use pytorch device_name: cpu
2025-11-02 21:46:35,166 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loaded 10 documents from ./docs
Docs: ['Cheetah', 'Dolphin', 'Elephant', 'Giraffe', 'Hippo', 'Kangaroo', 'Panda', 'Penguin', 'Tiger', 'Zebra']


/home/dsu/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Chunk Size | Accuracy
---------------------


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1024       | 1.00
